# Getting started
This is a working notebook for generating data analysis capsules for the FutureHouse Evals team. You'll find basic instructions for how to use this notebook below.

If you are not familiar with Colab Notebooks, please visit the welcome notebook at: https://colab.research.google.com/notebooks/intro.ipynb. In short, they function similarly to Jupyter Notebooks, but are more shareable for our purpose.

## Basic instructions
This notebook has been created for your specific use, so you should work directly in it.

You may write code in either `Python` or `R`, as well as use terminal commands. The default is `Python`, but you may use `R` on a cell-by-cell basis using the code example below, or in the whole notebook by changing the Runtime in the tab at the top of the notebook.

We ask that you are as verbose as possible in your descriptions of various steps using either in-line comments or text cells. *This includes reasoning about the steps taken and decisions made for using various tools or carrying out specific steps.*

It is also helpful if you are liberal in your use of code cells, e.g. by using one cell per function, rather than including too much code together in a single cell.

## File handling
Files accessed and written in this notebook environment are stored in `/content/`. This directory is viewable by clicking on the folder icon on the left, where you can upload any input files necessary for your analysis. These can be accessed simply with their filename, without the need to prefix with `/content/`.

Any files you upload as input data, *MUST ALSO* be uploaded to the same google drive folder that this notebook is contained in.

*DO NOT* use any google docs in your analysis (e.g. csv files turned into google sheets.) Use the original csv, txt, etc.

---

As always, if you have any questions do not hesitate to reach out to Jon via Slack or email (jon@futurehouse.org)

```
## Template version 2.1
```





---

## If using R, read below. Otherwise skip to [Analysis](#scrollTo=vA60UJdshKhw)

If your entire notebook will be `R`, you can change the notebook runtime in the Runtime tab above.

If you will be using `R` in only part of your analysis, you can run individual cells with R using the the steps below:

*   Run the cell below first to load the rmagic extension.
*   You can then include `%%R` at the top of any cell to run it as `R` code rather than `Python`.




---



# Analysis

In [ ]:
# Check R version
R.version.string

[1] "R version 4.4.2 (2024-10-31)"

### Package loading

In [ ]:
## Please install and load all external packages and libraries below this cell.

In [ ]:
# Loading required libraries
# This is for reading in excel files
if (!requireNamespace("readxl", quietly = TRUE)) {
  install.packages("readxl")
}
library(readxl)

# Deseq2,v1.32.0 on R/4.1.1 for differential expression analysis
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")
BiocManager::install("DESeq2")
library(DESeq2)

# apeglm for log fold shrinkage in DeSeq2 analysis
BiocManager::install("apeglm")
library(apeglm)

# Reading in libraries for data manipulation
if (!requireNamespace("tidyverse", quietly = TRUE)) {
  install.packages("tidyverse")
}
library(tidyverse)
if (!requireNamespace("data.table", quietly = TRUE)) {
  install.packages("data.table")
}
library(data.table)

# Load package for gene ontology analysis
BiocManager::install("clusterProfiler")
library(clusterProfiler)

# Load package for genome wide annotation for human genes
BiocManager::install("org.Hs.eg.db")
library(org.Hs.eg.db)

# Loading libraries for creating enrich plot of gene ontology enrichment
# Documentation here: https://rdrr.io/bioc/enrichplot/man/emapplot.html
if (!requireNamespace("enrichplot", quietly = TRUE)) {
  install.packages("enrichplot")
}
library(enrichplot)
if (!requireNamespace("ggnewscale", quietly = TRUE)) {
  install.packages("ggnewscale")
}
library(ggnewscale)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.20 (BiocManager 1.30.25), R 4.4.2 (2024-10-31)

Installing package(s) 'BiocVersion', 'DESeq2'

also installing the dependencies ‘formatR’, ‘UCSC.utils’, ‘GenomeInfoDbData’, ‘zlibbioc’, ‘abind’, ‘SparseArray’, ‘lambda.r’, ‘futile.options’, ‘GenomeInfoDb’, ‘XVector’, ‘S4Arrays’, ‘DelayedArray’, ‘futile.logger’, ‘snow’, ‘BH’, ‘S4Vectors’, ‘IRanges’, ‘GenomicRanges’, ‘SummarizedExperiment’, ‘BiocGenerics’, ‘Biobase’, ‘BiocParallel’, ‘matrixStats’, ‘locfit’, ‘MatrixGenerics’, ‘RcppArmadillo’


Old packages: 'curl', 'evaluate', 'foreign', 'MASS', 'openssl', 'pillar',
  'processx', 'Rcpp', 'rpart', 'testthat', 'xfun'

Loading required package: S4Vectors

Loading required package: stats4

Loading req

### Data loading

Include a brief description of each input dataset as a comment above, below, or following the loading line.

If you are using local input data files, they can be uploaded to the notebook environment by clicking on the folder icon on the left. They can then be accessed simply with their filename.

*You must also upload any input data files to the google folder that this notebook is in! These cannot be read by the notebook, but we need them for processing later.*

In [ ]:
## Please load any external input data or define locations for local input data below this cell. Read further guidelines above.

In [ ]:
## Raw Data
# RNA-seq data have been deposited in the NCBI Gene Expression Omnibus database (GEO GSE230696, SubSeries GSE230685) and processed with standard RNA-seq pipelines as outlined in the paper (PMID 37053013)

## Processed Data
# RNA-seq processed data have been deposited in the NCBI Gene Expression Omnibus database (GEO GSE230696, SubSeries GSE230685). The raw counts and DESeq2 normalised counts matrix are available for download.

# Reading in the featureCounts table generated from raw RNA-seq data as our "countdata" file
# This file is the count matrix with columns assigned to the samples, and the raw counts for each gene listed
# Uploaded this file to google drive for reference
countdata<-read.table("Issy_ASXL1_blood_featureCounts_GeneTable_final.txt",header=TRUE,row.names = 1)

# Reading in the sample identifiers table as our "coldata" file. Also known as metadata.
# Uploaded this file to google drive for reference
coldata<-data.frame(read_excel("Issy_ASXL1_blood_coldata_gender.xlsx"))

# Reference genome GencodeV31_GR38, gencode.v31.primary_assembly.genes available here: https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_31/
# Uploaded this file to google drive for reference
gencode_genes<-read.csv("gencode.v31.primary_assembly.genes.csv")
gencode_genes<-dplyr::select(gencode_genes,gene_name,gene_type,seqnames,hgnc_id,gene_id)
#This file will be used for gene ontology analysis as well

# Reference genome HUGO Gene Nomenclature Committee version HGNC_05-09-19 available here: https://www.genenames.org/download/
# Uploaded this file to google drive for reference
hgnc_ref<-read.delim("HGNC_05-09-19.txt")


### Analysis code

*Please run all code cells in place in this notebook in the google folder! We need cell outputs to ensure that notebooks are complete and functional.*

In [ ]:
## Begin analysis code below this cell.


In [ ]:
## Begin analysis code below this cell.

#Reformating the coldata sample identifiers dataframe
# Make the sample names the row names
rownames(coldata)<-coldata$sample
# Remove the sample column now that the sample names are embedded as the row names
coldata$sample<-NULL
# Defining samples as disease or control.
# Note: DESeq2 will automatically compare sample groups by alphabetical order, so if the disease group is alphabetically earlier than the control group (e.g. A for ASXL1), the comparison would be with ASXL1 as the baseline.
coldata$condition[coldata$condition=="ASXL1"]<-"disease"
coldata$condition[coldata$condition=="Control"]<-"control"
# Here we can remove samples if needed. Always document reasoning of why samples are removed, if any!
# Remove 1640 and 1641 (1640 - alcohol use disorder, 1641 - taking valproic acid). These affect gene expression.
countdata$MGD1640B<-NULL
countdata$MGD1641B<-NULL

# Format countdata file into count matrix format
countdata<-as.matrix(countdata)

# Generating the DESeq2 Data Set (dds) from countdata and coldata
# Design indicates how to model the samples and control for batch differences. In this case, we want to model the samples by condition (compare disease to control), and control for sample sex
dds<-DESeqDataSetFromMatrix(countData = countdata,colData = coldata,design = ~sex+condition)
# Run DESeq2
dds<-DESeq(dds)
# DESeq2 results, adjusted by correction method
resultsNames(dds)
# This will output options for comparing the dataset. In this case we are examining disease vs control
# For log fold shrinkage for visualization and ranking of genes, we use apeglm method for effect size shrinkage (Zhu, Ibrahim, and Love 2018), which improves on the previous estimator
# Zhu, A., Ibrahim, J.G., Love, M.I. (2018) Heavy-tailed prior distributions for sequence count data: removing the noise and preserving large differences. Bioinformatics. https://doi.org/10.1093/bioinformatics/bty895
# Extract results for specific comparison of choice
res<-lfcShrink(dds, coef="condition_disease_vs_control",type="apeglm")

# Convert results from DESeq2 object to data frame and convert rownames to gene column
# https://github.com/hbctraining/DGE_workshop/blob/master/lessons/05_DGE_DESeq2_analysis2.md
res<- res %>%
  data.frame() %>%
  rownames_to_column(var="gene_id") %>%
  as_tibble()

# Add gene nomenclature and gene IDs to the dataframe
res<-merge(res,gencode_genes,all.x=TRUE) # Merge res dataframe with gencode_genes dataframe
res<-merge(res,hgnc_ref,all.x=TRUE,by.x="hgnc_id",by.y="HGNC.ID") # Merge res + gencode_genes file with the hgnc_id dataframe

# Re-order the dataset by adjusted p value, and filter for padj < 0.05. These are the differentially expressed genes (DEGs).
# Can choose to filter for absolute log2 Fold Change values if desired. log2FC of 0.58 equals fold change of 1.5
final_res<-res%>%
  filter(padj<0.05)%>% # Filter by padj < 0.05
  #filter(abs(log2FoldChange)>0.58)%>%
  arrange(desc(abs(log2FoldChange))) # Arrange the dataframe rows by desending order of value of log2FoldChange
# This dataframe will be used for gene ontology analysis

# Generate gene ontology analysis for set of differentially expressed genes
# The gencode_genes dataframe will be used as the background i.e. "all genes"
# The set of genes to be tested are in the "final_res" dataframe i.e. genes that were significantly differentially expressed from DESeq2 analysis (padj < 0.05)
# Cleaning up dataframes to get rid of version numbers of gene IDs
all_genes<-gsub("\\..*","",gencode_genes$gene_id) #set of genes to use as background
sig_genes<-gsub("\\..*","",final_res$gene_id) #significant DEGs to be tested

# Gene ontology enrichment analysis of a gene set after FDR control
ego_all<-enrichGO(gene=sig_genes, #significant DEGs to be tested for GO enrichment
                  universe=all_genes, #full set of genes to use as the background universe
                  keyType = "ENSEMBL", #specifies that the gene identifiers are in ENSEMBL format
                  OrgDb = org.Hs.eg.db, #organism database of interest here is homo sapiens
                  ont = "ALL", #all gene ontology categories are considered i.e. Biological Processes, Molecular Function etc.
                  pAdjustMethod = "BH", #Benjamini-Hochberg correction for multiple testing used here to adjust for p values
                  qvalueCutoff = 0.05, #This is the threshold for the adjusted p value (q value)
                  readable = TRUE) #convert for easier reading

# Write out readable dataframe for summary of all gene ontology enrichment for this gene set
cluster_summary_all<-data.frame(ego_all) #converts our enrichment result into a readable data frame
#write.csv(cluster_summary_all,"GO_Issy_ASXL1_blood.csv") #option to save summary data frame in a csv file

# Gene ontology enrichment analysis of a gene set after FDR control, filtered by biological process (BP) ontologies
# https://guangchuangyu.github.io/2015/10/use-simplify-to-remove-redundancy-of-enriched-go-terms/
ego_bp<-enrichGO(gene=sig_genes, #testing the same set of significant DEGs as above
                 universe=all_genes, #testing against the same background set of all genes
                 keyType = "ENSEMBL", #specifies that we are using ENSEMBL gene ID notation
                 OrgDb = org.Hs.eg.db, #Homo sapiens annotation database
                 ont = "BP", #here we filter for only the biological process related gene ontology terms
                 pAdjustMethod = "BH", #Benjamini-Hochberg p value adjustment method
                 qvalueCutoff = 0.05, #adjusted p value cutoff for significance
                 readable = TRUE) #convert for easier reading

# Simplify our gene ontology results to reduce redundancy in GO terms
ego_bp_simp <- clusterProfiler::simplify(
  x = ego_bp, # Simplify the enriched GO terms from the analysis for only biological processes
  cutoff = 0.7, # GO terms with similarity > 0.7 are removed / simplified
  by = "p.adjust", # GO terms are ordered by their adjusted p-values to help prioritize which ones are of most interest
  select_fun = min # Select terms within a redundant cluster by picking the one with the "min" adjusted p value
)

# Write out readable dataframe for summary of biological process gene ontology enrichment for this gene set
summary_simp<-data.frame(ego_bp_simp)
#write.csv(summary_simp,"Simplified_GO_Issy_ASXL1_blood.csv")

# Print out top 10 most dysregulated gene ontologies
# Subset the data frame and take only the top 10 rows. We already sorted by padj so these are the most dysregulated gene ontologies.
subset_df <- summary_simp[, c("ID", "Description", "p.adjust")]
top_10 <- head(subset_df, 10) # Pick the first 10 rows and subset to new dataframe

# Print the top 10 most dysregulated gene ontologies with respective column names of gene ontology ID and padj values
for (i in 1:nrow(top_10)) {
  cat("Gene Ontology:", top_10$ID[i], "\n") # Take the values from the column "ID"
  cat("Description:", top_10$Description[i], "\n") # Take the values from the column "Description"
  cat("p.adj:", top_10$p.adjust[i], "\n") # Take the values from the column "padj"
  cat("\n")  # Add a blank line between rows
}

# Calculate pairwise similarity between our enriched and simplified GO terms
# This helps to visualise our clusters of related GO terms based on how similar they are to each other
pairwise <- pairwise_termsim(ego_bp_simp)
# Output tiff file with dimensions 10in x 10in and high quality print resolution
tiff("enrichMap_GO_Issy_ASXL1_blood.tiff", units="in", width=10, height=10, res=300)
# Plot our enrichment map using the pairwise GO term similarities
emapplot(pairwise,cex_label_category=1) # category labels set as 1 i.e. visible
dev.off() #close off our TIFF device to save our plot


Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



[1] "Intercept"                    "sex_M_vs_F"                  
[3] "condition_disease_vs_control"

using 'apeglm' for LFC shrinkage. If used in published research, please cite:
    Zhu, A., Ibrahim, J.G., Love, M.I. (2018) Heavy-tailed prior distributions for
    sequence count data: removing the noise and preserving large differences.
    Bioinformatics. https://doi.org/10.1093/bioinformatics/bty895



Gene Ontology: GO:0030098 
Description: lymphocyte differentiation 
p.adj: 6.963017e-06 

Gene Ontology: GO:0002768 
Description: immune response-regulating cell surface receptor signaling pathway 
p.adj: 1.847038e-05 

Gene Ontology: GO:0002274 
Description: myeloid leukocyte activation 
p.adj: 4.661872e-05 

Gene Ontology: GO:0007409 
Description: axonogenesis 
p.adj: 7.820659e-05 

Gene Ontology: GO:0050867 
Description: positive regulation of cell activation 
p.adj: 0.0002002892 

Gene Ontology: GO:0050863 
Description: regulation of T cell activation 
p.adj: 0.0002045181 

Gene Ontology: GO:0032943 
Description: mononuclear cell proliferation 
p.adj: 0.0003246646 

Gene Ontology: GO:0046651 
Description: lymphocyte proliferation 
p.adj: 0.0003246646 

Gene Ontology: GO:0006935 
Description: chemotaxis 
p.adj: 0.0003246646 

Gene Ontology: GO:0010975 
Description: regulation of neuron projection development 
p.adj: 0.0003481683 



ERROR: Error in emapplot_internal(x, showCategory = showCategory, ...): unused argument (cex_label_category = 1)
